# NB_bishop_ch10_figures

**Bishop Chapter 10 — Key Figures: 1D/2D Convolution, Pooling, CNN Architecture, Feature Maps & Receptive Field**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 10 on convolutional networks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_10/NB_bishop_ch10_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import Normalize
from scipy.signal import convolve2d
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 10.2 — 1D Convolution (Stem Plots)

In [ ]:
np.random.seed(42)

# Input signal
n = 20
x_signal = np.zeros(n)
x_signal[3:7]   = [1, 2, 3, 2]
x_signal[10:14]  = [2, 3, 2, 1]
x_signal[16:19]  = [1, 1.5, 0.5]

# Kernel (edge detector)
kernel = np.array([1, 0, -1], dtype=float)

# Full convolution
output = np.convolve(x_signal, kernel, mode='same')

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=False)

# Input
markerline, stemlines, baseline = axes[0].stem(np.arange(n), x_signal)
plt.setp(stemlines, color=COLORS['blue'], lw=1.5)
plt.setp(markerline, color=COLORS['blue'], ms=6)
plt.setp(baseline, color='gray', lw=0.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Input Signal $x[n]$')

# Kernel
k_idx = np.arange(len(kernel))
markerline, stemlines, baseline = axes[1].stem(k_idx, kernel)
plt.setp(stemlines, color=COLORS['red'], lw=2)
plt.setp(markerline, color=COLORS['red'], ms=8)
plt.setp(baseline, color='gray', lw=0.5)
axes[1].set_ylabel('Weight')
axes[1].set_title('Kernel $w[k] = [1, 0, -1]$ (edge detector)')
axes[1].set_xlim(-0.5, 2.5)

# Output
markerline, stemlines, baseline = axes[2].stem(np.arange(n), output)
plt.setp(stemlines, color=COLORS['green'], lw=1.5)
plt.setp(markerline, color=COLORS['green'], ms=6)
plt.setp(baseline, color='gray', lw=0.5)
axes[2].set_xlabel('Sample index $n$')
axes[2].set_ylabel('Amplitude')
axes[2].set_title('Output $(x * w)[n]$')

fig.tight_layout()
save_fig(fig, 'fig10_2_convolution_1d')
plt.show()

## Figure 10.3 — 2D Convolution (Annotated Grids)

In [ ]:
# 5x5 input
inp = np.array([
    [1, 0, 2, 3, 1],
    [0, 1, 3, 1, 0],
    [2, 3, 0, 2, 1],
    [1, 2, 1, 0, 3],
    [0, 1, 2, 3, 2]
], dtype=float)

# 3x3 kernel
kern = np.array([
    [ 1,  0, -1],
    [ 1,  0, -1],
    [ 1,  0, -1]
], dtype=float)

# Valid convolution output (3x3)
out = convolve2d(inp, kern[::-1, ::-1], mode='valid')  # cross-correlation

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

def draw_grid(ax, data, title, cmap='Blues', highlight=None, highlight_color=None):
    """Draw annotated grid with cell values."""
    rows, cols = data.shape
    ax.imshow(data, cmap=cmap, vmin=data.min() - 1, vmax=data.max() + 1)
    for i in range(rows):
        for j in range(cols):
            ax.text(j, i, f'{data[i, j]:.0f}', ha='center', va='center',
                    fontsize=13, fontweight='bold')
    # Highlight receptive field
    if highlight is not None:
        r, c = highlight
        rect = Rectangle((c - 0.5, r - 0.5), 3, 3, lw=3,
                         edgecolor=highlight_color or COLORS['red'],
                         facecolor='none', zorder=5)
        ax.add_patch(rect)
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_title(title, fontsize=12)

# Input with receptive field highlighted
draw_grid(axes[0], inp, 'Input (5 x 5)', cmap='Blues', highlight=(1, 1),
          highlight_color=COLORS['red'])
axes[0].set_xlabel('Receptive field for output (1,1)', fontsize=9, color=COLORS['red'])

# Kernel
draw_grid(axes[1], kern, 'Kernel (3 x 3)', cmap='RdBu_r')

# Output with computed cell highlighted
draw_grid(axes[2], out, 'Output (3 x 3)', cmap='Greens')
rect = Rectangle((1 - 0.5, 1 - 0.5), 1, 1, lw=3,
                 edgecolor=COLORS['red'], facecolor='none', zorder=5)
axes[2].add_patch(rect)

# Arrow between panels
fig.text(0.365, 0.5, '$*$', fontsize=28, ha='center', va='center',
         fontweight='bold', color=COLORS['amber'])
fig.text(0.66, 0.5, '$=$', fontsize=28, ha='center', va='center',
         fontweight='bold', color=COLORS['amber'])

fig.suptitle('2D Convolution (Cross-Correlation)', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig10_3_convolution_2d')
plt.show()

## Figure 10.8 — Max Pooling and Average Pooling

In [ ]:
# 4x4 input feature map
pool_input = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 2, 7, 0],
    [1, 4, 5, 3]
], dtype=float)

# 2x2 pooling
def max_pool_2x2(x):
    h, w = x.shape
    out = np.zeros((h // 2, w // 2))
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i // 2, j // 2] = x[i:i+2, j:j+2].max()
    return out

def avg_pool_2x2(x):
    h, w = x.shape
    out = np.zeros((h // 2, w // 2))
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            out[i // 2, j // 2] = x[i:i+2, j:j+2].mean()
    return out

max_out = max_pool_2x2(pool_input)
avg_out = avg_pool_2x2(pool_input)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Colour the 2x2 quadrants in the input
quad_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['amber']]

def draw_pool_grid(ax, data, title, cmap_vals=None):
    rows, cols = data.shape
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)
    ax.set_aspect('equal')
    for i in range(rows):
        for j in range(cols):
            fc = 'white'
            if cmap_vals is not None:
                ci = (i // (rows // 2)) * 2 + (j // (cols // 2)) if rows == 4 else i * 2 + j
                fc = matplotlib.colors.to_rgba(quad_colors[ci % 4], alpha=0.15)
            rect = Rectangle((j - 0.5, i - 0.5), 1, 1, fc=fc, ec='gray', lw=1)
            ax.add_patch(rect)
            ax.text(j, i, f'{data[i, j]:.1f}' if data[i, j] != int(data[i, j])
                    else f'{int(data[i, j])}',
                    ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_title(title, fontsize=12)

# Input
draw_pool_grid(axes[0], pool_input, 'Input (4 x 4)', cmap_vals='quad')
# Draw quadrant borders
for pos in [1.5]:
    axes[0].axhline(pos, color='k', lw=2)
    axes[0].axvline(pos, color='k', lw=2)

# Max pooling output
draw_pool_grid(axes[1], max_out, 'Max Pool 2x2 (2 x 2)', cmap_vals='quad')

# Avg pooling output
draw_pool_grid(axes[2], avg_out, 'Avg Pool 2x2 (2 x 2)', cmap_vals='quad')

# Arrows
fig.text(0.365, 0.5, 'max', fontsize=14, ha='center', va='center',
         color=COLORS['blue'], fontweight='bold')
fig.text(0.365, 0.38, '$\\longrightarrow$', fontsize=20, ha='center', va='center',
         color=COLORS['blue'])

fig.text(0.66, 0.5, 'avg', fontsize=14, ha='center', va='center',
         color=COLORS['amber'], fontweight='bold')
fig.text(0.66, 0.38, '$\\longrightarrow$', fontsize=20, ha='center', va='center',
         color=COLORS['amber'])

fig.suptitle('Pooling Operations (stride 2)', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig10_8_pooling')
plt.show()

## Figure 10 — CNN Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.set_xlim(-0.5, 14)
ax.set_ylim(-1.5, 3.5)
ax.set_aspect('equal')
ax.axis('off')

# ── layer specifications ────────────────────────────────────
layers = [
    {'x': 0.0, 'w': 1.0, 'h': 2.8, 'label': 'Input\n32x32x3',  'fc': '#e8edf3', 'ec': COLORS['blue']},
    {'x': 2.0, 'w': 0.8, 'h': 2.4, 'label': 'Conv\n3x3, 32',   'fc': '#fff3e0', 'ec': COLORS['amber']},
    {'x': 3.5, 'w': 0.6, 'h': 1.8, 'label': 'Pool\n2x2',       'fc': '#e8f5e9', 'ec': COLORS['green']},
    {'x': 5.0, 'w': 0.8, 'h': 1.8, 'label': 'Conv\n3x3, 64',   'fc': '#fff3e0', 'ec': COLORS['amber']},
    {'x': 6.5, 'w': 0.6, 'h': 1.2, 'label': 'Pool\n2x2',       'fc': '#e8f5e9', 'ec': COLORS['green']},
    {'x': 8.2, 'w': 0.3, 'h': 2.5, 'label': 'Flatten',         'fc': '#f3e5f5', 'ec': '#7B1FA2'},
    {'x': 9.5, 'w': 0.4, 'h': 2.0, 'label': 'FC\n256',         'fc': '#e3f2fd', 'ec': COLORS['blue']},
    {'x': 11.0,'w': 0.4, 'h': 1.2, 'label': 'FC\n10',          'fc': '#fce4ec', 'ec': COLORS['red']},
    {'x': 12.5,'w': 0.4, 'h': 1.0, 'label': 'Softmax\nOutput', 'fc': '#fce4ec', 'ec': COLORS['red']},
]

for i, L in enumerate(layers):
    y_bot = (3.0 - L['h']) / 2  # center vertically
    rect = FancyBboxPatch((L['x'], y_bot), L['w'], L['h'],
                          boxstyle='round,pad=0.05', fc=L['fc'], ec=L['ec'], lw=2)
    ax.add_patch(rect)
    ax.text(L['x'] + L['w'] / 2, y_bot + L['h'] / 2, L['label'],
            ha='center', va='center', fontsize=8, fontweight='bold',
            color=L['ec'])

    # Arrow to next layer
    if i < len(layers) - 1:
        nxt = layers[i + 1]
        ax.annotate('', xy=(nxt['x'] - 0.08, 1.5),
                    xytext=(L['x'] + L['w'] + 0.08, 1.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                                   mutation_scale=12))

# Group labels
ax.text(3.0, -0.8, 'Feature Extraction', fontsize=11, ha='center',
        color=COLORS['amber'], style='italic')
ax.annotate('', xy=(0.0, -0.55), xytext=(7.1, -0.55),
            arrowprops=dict(arrowstyle='|-|', color=COLORS['amber'], lw=1))

ax.text(10.8, -0.8, 'Classification', fontsize=11, ha='center',
        color=COLORS['red'], style='italic')
ax.annotate('', xy=(8.2, -0.55), xytext=(12.9, -0.55),
            arrowprops=dict(arrowstyle='|-|', color=COLORS['red'], lw=1))

ax.set_title('CNN Architecture', fontsize=14, pad=10)

fig.tight_layout()
save_fig(fig, 'fig10_cnn_architecture')
plt.show()

## Figure 10 — Feature Maps (Edge Detection Kernels)

In [ ]:
# Create a simple test image: checkerboard + cross pattern
img = np.zeros((32, 32))
# Checkerboard quadrants
img[:16, :16] = 1.0
img[16:, 16:] = 1.0
# Add a cross
img[14:18, :] = 0.5
img[:, 14:18] = 0.5
# Add diagonal stripe
for k in range(-2, 3):
    d = np.diag(np.ones(32 - abs(k)), k)
    img += 0.3 * d[:32, :32]
img = np.clip(img, 0, 1)

# Edge detection kernels
kernels = {
    'Horizontal': np.array([[-1, -1, -1],
                            [ 0,  0,  0],
                            [ 1,  1,  1]], dtype=float),
    'Vertical':   np.array([[-1, 0, 1],
                            [-1, 0, 1],
                            [-1, 0, 1]], dtype=float),
    'Diagonal (\\\\)':  np.array([[ 0,  1,  1],
                             [-1,  0,  1],
                             [-1, -1,  0]], dtype=float),
    'Diagonal (/)': np.array([[ 1,  1,  0],
                              [ 1,  0, -1],
                              [ 0, -1, -1]], dtype=float),
}

fig, axes = plt.subplots(2, 5, figsize=(15, 6),
                         gridspec_kw={'width_ratios': [1, 1, 1, 1, 1]})

# Top row: original + kernels
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Input Image', fontsize=10)
axes[0, 0].axis('off')

for idx, (name, k) in enumerate(kernels.items()):
    ax_k = axes[0, idx + 1]
    ax_k.imshow(k, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    for i in range(3):
        for j in range(3):
            ax_k.text(j, i, f'{int(k[i,j])}', ha='center', va='center',
                      fontsize=12, fontweight='bold')
    ax_k.set_title(f'{name}\nKernel', fontsize=9)
    ax_k.axis('off')

# Bottom row: empty + convolution results
axes[1, 0].axis('off')

cmap_out = 'RdBu_r'
for idx, (name, k) in enumerate(kernels.items()):
    result = convolve2d(img, k, mode='same', boundary='fill')
    vmax = max(abs(result.min()), abs(result.max()))
    axes[1, idx + 1].imshow(result, cmap=cmap_out, vmin=-vmax, vmax=vmax)
    axes[1, idx + 1].set_title(f'{name}\nResponse', fontsize=9)
    axes[1, idx + 1].axis('off')

fig.suptitle('Edge Detection Feature Maps', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig10_feature_maps')
plt.show()

## Figure 10 — Receptive Field Growth Across Layers

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 15)
ax.set_aspect('equal')
ax.axis('off')

# Represent an input feature map as a grid
grid_size = 14
for i in range(grid_size + 1):
    ax.plot([0, grid_size], [i, i], color='lightgray', lw=0.5)
    ax.plot([i, i], [0, grid_size], color='lightgray', lw=0.5)

# Receptive field sizes for stacked 3x3 convolutions
# Layer 1: 3x3, Layer 2: 5x5, Layer 3: 7x7, Layer 4: 9x9
rf_specs = [
    {'size': 9, 'color': COLORS['blue'],  'alpha': 0.10, 'label': 'Layer 4: 9x9 RF'},
    {'size': 7, 'color': COLORS['green'], 'alpha': 0.12, 'label': 'Layer 3: 7x7 RF'},
    {'size': 5, 'color': COLORS['amber'], 'alpha': 0.15, 'label': 'Layer 2: 5x5 RF'},
    {'size': 3, 'color': COLORS['red'],   'alpha': 0.20, 'label': 'Layer 1: 3x3 RF'},
]

center = grid_size / 2

legend_handles = []
for spec in rf_specs:
    s = spec['size']
    offset = center - s / 2
    rect = Rectangle((offset, offset), s, s,
                     fc=matplotlib.colors.to_rgba(spec['color'], spec['alpha']),
                     ec=spec['color'], lw=2.5, ls='-')
    ax.add_patch(rect)
    legend_handles.append(mpatches.Patch(facecolor=spec['color'], alpha=0.3,
                                         edgecolor=spec['color'], label=spec['label']))

# Mark center pixel
ax.plot(center, center, 's', color='k', ms=8, zorder=10)
ax.text(center + 0.5, center + 0.5, 'Output\nneuron', fontsize=9, va='bottom')

ax.legend(handles=legend_handles)

ax.set_title('Receptive Field Growth with Stacked 3x3 Convolutions', fontsize=13, pad=15)
ax.text(7, -0.7, 'Each additional 3x3 conv layer adds +2 to the receptive field',
        ha='center', fontsize=10, style='italic', color='gray')

fig.tight_layout()
save_fig(fig, 'fig10_receptive_field')
plt.show()

In [ ]:
print('All Chapter 10 figures generated.')